# Legal Entity Recognition (LER) — Baseline Models
## Putusan Pengujian Undang-Undang (PUU) Mahkamah Konstitusi RI

Notebook ini membangun dan mengevaluasi tiga model baseline:
1. **CRF** (Conditional Random Field)
2. **BiLSTM** (Bidirectional LSTM)
3. **BiLSTM-CRF** (Combined Architecture)

### Label Set
```
PERSON, ORG, JUDGE, DECISION_NUM, LAW_NAME, DATE, EVENT,
DICTUM, REGISTRAR, LAWYER, LAW_CONS_ART, PARTY_REF,
LEGAL_DOC, DECISION_DATE, LEGAL_DOC_NUM
```

### Format Data
File CoNLL BIO: `train.conll`, `dev.conll`, `test.conll`
```
Token  B-LABEL / I-LABEL / O
```

---
## Alur Notebook

| # | Bagian | Isi |
|---|--------|-----|
| 0 | Instalasi | Library yang dibutuhkan |
| 1 | Import & Konfigurasi Global | Semua hyperparameter terpusat di sini |
| 2 | Load & Eksplorasi Data | Baca CoNLL, statistik dataset |
| 3 | Vocabulary Bersama | Token & label vocab (dipakai ketiga model) |
| 4 | Model 1 — CRF | Feature engineering, training, evaluasi |
| 5 | Model 2 — BiLSTM | Dataset PyTorch, arsitektur, training, evaluasi |
| 6 | Model 3 — BiLSTM-CRF | CRF layer, arsitektur gabungan, training, evaluasi |
| 7 | Perbandingan & Visualisasi | Tabel, bar chart, learning curves |
| 8 | Analisis Per-Label | Heatmap F1 per entitas |
| 9 | Simpan Model & Hasil | Pickle / .pt / CSV |
| 10 | Inference Demo | Contoh prediksi kalimat baru |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 0. Instalasi Dependensi

In [ ]:
!pip install sklearn-crfsuite seqeval torch numpy pandas matplotlib scikit-learn tqdm pytorch-crf -q
print("✅ Instalasi selesai.")

---
## 1. Import & Konfigurasi Global

> **Semua hyperparameter diatur di sini.** BiLSTM dan BiLSTM-CRF berbagi arsitektur
> yang sama (Embedding + BiLSTM), sehingga `EMBEDDING_DIM`, `HIDDEN_DIM`, `NUM_LAYERS`,
> `DROPOUT`, `BATCH_SIZE`, `PATIENCE` dapat dipakai bersama.
> Namun **`LEARNING_RATE` dipisah** karena lapisan CRF pada BiLSTM-CRF membutuhkan
> learning rate yang lebih kecil agar transition matrix-nya konvergen stabil.

In [ ]:
import os, random, json, time, warnings, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import classification_report

# CRF
import sklearn_crfsuite
from sklearn_crfsuite import metrics as crf_metrics

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torchcrf import CRF

# Seqeval
from seqeval.metrics import (
    classification_report as seq_classification_report,
    f1_score        as seq_f1,
    precision_score as seq_precision,
    recall_score    as seq_recall,
)

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# ── Path data ──────────────────────────────────────────────────────────────────
DATA_DIR   = '/content/drive/MyDrive/data_conll_100_v6.1'   # sesuaikan
TRAIN_FILE = os.path.join(DATA_DIR, 'train.conll')
DEV_FILE   = os.path.join(DATA_DIR, 'dev.conll')
TEST_FILE  = os.path.join(DATA_DIR, 'test.conll')

# ══════════════════════════════════════════════════════════════════════════════
#  KONFIGURASI BERSAMA  (BiLSTM & BiLSTM-CRF)
# ══════════════════════════════════════════════════════════════════════════════
EMBEDDING_DIM = 300    # dimensi embedding token
HIDDEN_DIM    = 512    # hidden size BiLSTM (per arah = HIDDEN_DIM // 2)
NUM_LAYERS    = 3      # jumlah layer BiLSTM
DROPOUT       = 0.25   # dropout embedding & antar layer LSTM
BATCH_SIZE    = 16     # ukuran batch training
MAX_EPOCHS    = 100    # epoch maksimum (early stopping bisa menghentikan lebih cepat)
PATIENCE      = 20     # early stopping: berhenti jika dev F1 tidak naik selama N epoch
WEIGHT_DECAY  = 1e-4   # L2 regularization (AdamW)

# ── Learning Rate TERPISAH per model ──────────────────────────────────────────
# BiLSTM murni: LR bisa lebih besar karena output langsung ke Softmax
BILSTM_LR     = 5e-4

# BiLSTM-CRF: LR lebih kecil agar transition matrix CRF konvergen stabil
# CRF menambah parameter (num_labels × num_labels) dengan skala gradien berbeda
BILSTMCRF_LR  = 1e-4

# ── Differential LR untuk BiLSTM-CRF ─────────────────────────────────────────
BILSTMCRF_LR_LSTM = 3e-4   # untuk Embedding + BiLSTM + Linear
BILSTMCRF_LR_CRF  = 2e-5   # untuk CRF transition matrix (2× lebih kecil)

print("✅ Konfigurasi global selesai.")
print(f"   EMBEDDING_DIM = {EMBEDDING_DIM} | HIDDEN_DIM = {HIDDEN_DIM} | NUM_LAYERS = {NUM_LAYERS}")
print(f"   DROPOUT = {DROPOUT} | BATCH_SIZE = {BATCH_SIZE} | PATIENCE = {PATIENCE}")
print(f"   BILSTM_LR = {BILSTM_LR} | BILSTMCRF_LR = {BILSTMCRF_LR}")

---
## 2. Load & Eksplorasi Data CoNLL

In [ ]:
def read_conll(filepath):
    """Membaca file CoNLL BIO. Mengembalikan list of (tokens, labels)."""
    sentences, tokens, labels = [], [], []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            if line == '' or line.startswith('-DOCSTART-'):
                if tokens:
                    sentences.append((tokens, labels))
                    tokens, labels = [], []
            else:
                parts = line.split()
                tokens.append(parts[0])
                labels.append(parts[-1])
    if tokens:
        sentences.append((tokens, labels))
    return sentences


def dataset_stats(name, dataset):
    """Cetak statistik dasar dataset."""
    all_labels    = [l for _, labels in dataset for l in labels]
    entity_labels = [l for l in all_labels if l != 'O']
    label_counts  = Counter(all_labels)

    print(f"\n{'─'*50}\n {name} Dataset\n{'─'*50}")
    print(f"  Kalimat   : {len(dataset):,}")
    print(f"  Token     : {len(all_labels):,}")
    print(f"  Entity tok: {len(entity_labels):,} ({len(entity_labels)/len(all_labels)*100:.1f}%)")
    b_counts = {k: v for k, v in label_counts.items() if k.startswith('B-')}
    for label, count in sorted(b_counts.items(), key=lambda x: -x[1]):
        print(f"  {label:<25} {count:>6}")


train_data = read_conll(TRAIN_FILE)
dev_data   = read_conll(DEV_FILE)
test_data  = read_conll(TEST_FILE)

dataset_stats('Train', train_data)
dataset_stats('Dev',   dev_data)
dataset_stats('Test',  test_data)
print("\n✅ Data berhasil dimuat.")

---
## 3. Vocabulary & Utilitas Bersama

Vocabulary token dan label dibangun **sekali** dari training set dan dipakai oleh
ketiga model (CRF, BiLSTM, BiLSTM-CRF) agar evaluasi perbandingan konsisten.

In [ ]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

all_train_tokens = [t for tokens, _ in train_data for t in tokens]
all_train_labels = [l for _, labels in train_data for l in labels]

# ── Label vocabulary — urutan HARUS konsisten ──────────────────────────────────
ALL_LABELS = [
    'O',
    'B-DATE',           'I-DATE',
    'B-DECISION_DATE',  'I-DECISION_DATE',
    'B-DECISION_NUM',   'I-DECISION_NUM',
    'B-DICTUM_BODY',    'I-DICTUM_BODY',
    'B-DICTUM_HEAD',
    'B-EVENT',          'I-EVENT',
    'B-JUDGE',          'I-JUDGE',
    'B-LAWYER',         'I-LAWYER',
    'B-LAW_CONS_ART',   'I-LAW_CONS_ART',
    'B-LAW_NAME',       'I-LAW_NAME',
    'B-LEGAL_DOC',      'I-LEGAL_DOC',
    'B-LEGAL_DOC_NUM',  'I-LEGAL_DOC_NUM',
    'B-ORG',            'I-ORG',
    'B-PARTY_REF',      'I-PARTY_REF',
    'B-PERSON',         'I-PERSON',
    'B-REGISTRAR',      'I-REGISTRAR',
]

label2idx     = {l: i for i, l in enumerate(ALL_LABELS)}
idx2label     = {i: l for l, i in label2idx.items()}
NUM_LABELS    = len(ALL_LABELS)          # 32
PAD_LABEL_IDX = NUM_LABELS               # 32 — di luar range label valid

# ── Token vocabulary ───────────────────────────────────────────────────────────
word_counter = Counter(all_train_tokens)
vocab        = [PAD_TOKEN, UNK_TOKEN] + [w for w, _ in word_counter.most_common()]
word2idx     = {w: i for i, w in enumerate(vocab)}
idx2word     = {i: w for w, i in word2idx.items()}
VOCAB_SIZE   = len(vocab)
PAD_IDX      = word2idx[PAD_TOKEN]   # 0
UNK_IDX      = word2idx[UNK_TOKEN]   # 1

# ── Char vocabulary (untuk fitur CRF) ─────────────────────────────────────────
all_chars  = set(c for t in all_train_tokens for c in t)
char2idx   = {c: i+2 for i, c in enumerate(sorted(all_chars))}
char2idx[PAD_TOKEN] = 0
char2idx[UNK_TOKEN] = 1

print(f"Vocab size    : {VOCAB_SIZE:,}")
print(f"Char vocab    : {len(char2idx):,}")
print(f"Num labels    : {NUM_LABELS}")
print(f"PAD_LABEL_IDX : {PAD_LABEL_IDX}  (di luar label valid)")
print(f"Index 'O'     : {label2idx['O']}")
print("\n✅ Vocabulary selesai.")

---
## 4. Model 1 — CRF (Conditional Random Field)
### 4.1 Feature Engineering

In [ ]:
MAX_SEQ_LEN  = 80

DICTUM_BODY_TRIGGERS  = {'mengadili','amar','mengabulkan','menolak','menyatakan','memerintahkan','memutuskan'}
DICTUM_RESET_TRIGGERS = {'mendengar','membaca','memeriksa','menimbang'}
JUDGE_CONTEXT   = {'hakim','ketua','anggota','majelis','konstitusi','mahkamah','merangkap'}
LAWYER_CONTEXT  = {'kuasa','hukum','advokat','pengacara','firma','law'}
PERSON_CONTEXT  = {'pemohon','termohon','terkait','saksi','ahli','warga','negara','pihak','intervensi'}
GELAR_TOKENS    = {'dr','prof','drs','ir','drg','s.h','m.h','ll.m','sh','mh','s.e','m.m',
                   'm.si','m.hum','dfm','ph.d','m.kn','m.pd','m.eng','msc','ak',
                   'sp.a','sp.kk','sp.p','sp.og','sp.d','kep','a.md','lc','bin','binti'}
SECTION_MARKERS = {'KEPALA','PIHAK','DUDUK_PERKARA','PERTIMBANGAN','AMAR','PENUTUP'}

RE_AMAR_NUM  = re.compile(r'^\d+\.$')
RE_GELAR_DOT = re.compile(r'^[A-Z][a-z]?\.[A-Z]')

def normalize(token):
    return token.lower().rstrip('.,;:')

def get_section_tags(tokens):
    sections, current_section, i = [], 'UNKNOWN', 0
    while i < len(tokens):
        if (tokens[i] == '===' and i+2 < len(tokens)
                and tokens[i+1] in SECTION_MARKERS and tokens[i+2] == '==='):
            current_section = tokens[i+1]
            sections.extend([current_section]*3)
            i += 3
        else:
            sections.append(current_section)
            i += 1
    return sections

def get_role_context(sent, i, window=6):
    ctx = {normalize(sent[k]) for k in range(max(0,i-window), min(len(sent),i+window+1))}
    return {'ctx_has_judge': bool(ctx & JUDGE_CONTEXT),
            'ctx_has_lawyer': bool(ctx & LAWYER_CONTEXT),
            'ctx_has_person': bool(ctx & PERSON_CONTEXT)}

def word_features(sent, i, sections=None):
    word, n = sent[i], len(sent)
    section  = sections[i] if sections else 'UNKNOWN'
    role_ctx = get_role_context(sent, i)
    rel_pos  = i / n
    ctx_lower = [sent[k].lower() for k in range(max(0,i-5), min(n,i+6))]
    in_dictum_context = any(t in DICTUM_BODY_TRIGGERS for t in ctx_lower)
    sent_has_trigger  = any(t.lower() in DICTUM_BODY_TRIGGERS for t in sent)

    f = {
        'bias': 1.0, 'word.lower': word.lower(),
        'word[-3:]': word[-3:], 'word[-2:]': word[-2:],
        'word[:3]': word[:3],   'word[:2]': word[:2],
        'word.isupper': word.isupper(), 'word.istitle': word.istitle(),
        'word.isdigit': word.isdigit(), 'word.has_hyphen': '-' in word,
        'word.has_slash': '/' in word,  'word.has_dot': '.' in word,
        'word.has_upper': any(c.isupper() for c in word), 'word.length': len(word),
        'word.is_roman': all(c in 'IVXLCDM' for c in word.upper()) and len(word)>0,
        'BOS': i == 0, 'EOS': i == n-1,
        'ctx.judge_nearby': role_ctx['ctx_has_judge'],
        'ctx.lawyer_nearby': role_ctx['ctx_has_lawyer'],
        'ctx.person_nearby': role_ctx['ctx_has_person'],
        'word.is_gelar': normalize(word) in GELAR_TOKENS,
        'word.has_gelar_dot': bool(RE_GELAR_DOT.match(word)),
        'word.is_name_like': word.istitle() and len(word)>2 and word.isalpha(),
        'word.is_dictum_trigger': word.lower() in DICTUM_BODY_TRIGGERS,
        'word.is_MENGADILI': word.upper()=='MENGADILI',
        'word.is_AMAR': word.upper()=='AMAR',
        'sent.has_trigger': sent_has_trigger,
        'in_dictum_context': in_dictum_context,
        'rel_pos_bin': str(int(rel_pos*5)),
        'word.is_amar_num': bool(RE_AMAR_NUM.match(word)),
        'word.is_semicolon': word==';', 'word.is_colon': word==':',
        'section': section,
        'section_allows_person': section in {'PIHAK','DUDUK_PERKARA','PERTIMBANGAN','UNKNOWN'},
        'section_is_dictum': section=='AMAR',
        'section_is_closing': section=='PENUTUP',
        'is_unknown_section': section=='UNKNOWN',
    }
    if i > 0:
        w_1 = sent[i-1]
        f.update({'-1:word.lower': w_1.lower(), '-1:word.istitle': w_1.istitle(),
                  '-1:word.isupper': w_1.isupper(), '-1:word[-3:]': w_1[-3:],
                  '-1:word.is_trigger': w_1.lower() in DICTUM_BODY_TRIGGERS,
                  '-1:section': sections[i-1] if sections else 'UNKNOWN'})
    if i >= 2:
        w_2 = sent[i-2]
        f.update({'-2:word.lower': w_2.lower(), '-2:word.istitle': w_2.istitle()})
    if i < n-1:
        w1 = sent[i+1]
        f.update({'+1:word.lower': w1.lower(), '+1:word.istitle': w1.istitle(),
                  '+1:word.isupper': w1.isupper(), '+1:word[-3:]': w1[-3:],
                  '+1:word.is_trigger': w1.lower() in DICTUM_BODY_TRIGGERS,
                  '+1:section': sections[i+1] if sections else 'UNKNOWN'})
    if i < n-2:
        w2 = sent[i+2]
        f.update({'+2:word.lower': w2.lower(), '+2:word.istitle': w2.istitle()})
    return f

def sent2features(sent):
    sections = get_section_tags(sent)
    return [word_features(sent, i, sections) for i in range(len(sent))]

def prepare_crf_data(dataset):
    X = [sent2features(tokens) for tokens, _ in dataset]
    y = [labels for _, labels in dataset]
    return X, y

print("Menyiapkan fitur CRF...")
X_train, y_train = prepare_crf_data(train_data)
X_dev,   y_dev   = prepare_crf_data(dev_data)
X_test,  y_test  = prepare_crf_data(test_data)
print(f"  Train: {len(X_train):,} | Dev: {len(X_dev):,} | Test: {len(X_test):,} kalimat")
print("✅ Fitur CRF siap.")

### 4.2 Pelatihan Model CRF

In [ ]:
print("Melatih CRF...")
t0_crf = time.time()

crf_model = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.05,
    c2=0.25,
    max_iterations=150,
    all_possible_transitions=False,
    all_possible_states=True,
)
crf_model.fit(X_train, y_train)
train_time_crf = time.time() - t0_crf

# Evaluasi dev set
y_dev_pred_crf = crf_model.predict(X_dev)
dev_f1_crf     = seq_f1(y_dev, y_dev_pred_crf)

print(f"✅ CRF selesai dalam {train_time_crf:.1f} detik | Dev F1: {dev_f1_crf:.4f}")

### 4.3 Evaluasi CRF pada Test Set

In [ ]:
y_test_pred_crf = crf_model.predict(X_test)

print("="*60)
print(" HASIL EVALUASI — CRF (Test Set)")
print("="*60)
print(seq_classification_report(y_test, y_test_pred_crf, digits=4))

crf_results = {
    'model':      'CRF',
    'precision':  seq_precision(y_test, y_test_pred_crf),
    'recall':     seq_recall(y_test, y_test_pred_crf),
    'f1':         seq_f1(y_test, y_test_pred_crf),
    'train_time': train_time_crf,
}
print(f"Macro P/R/F1 : {crf_results['precision']:.4f} / {crf_results['recall']:.4f} / {crf_results['f1']:.4f}")
print(f"Waktu latih  : {train_time_crf:.1f} detik")

### 4.4 Analisis Transisi & Fitur CRF (Opsional)

In [ ]:
def print_top_transitions(model, n=10):
    trans = [(s, f'{f} → {t}') for (f,t),s in model.transition_features_.items()]
    trans.sort(reverse=True)
    print("\n--- Top Positif Transisi ---")
    for score, pair in trans[:n]:   print(f"  {pair:<40} {score:+.4f}")
    print("\n--- Top Negatif Transisi ---")
    for score, pair in trans[-n:]:  print(f"  {pair:<40} {score:+.4f}")

def print_top_state_features(model, n=15):
    state = []
    for (feat, label), score in model.state_features_.items():
        desc = f"{feat} → {label}"
        state.append((score, desc))
    state.sort(reverse=True)
    print("\n--- Top State Features ---")
    for score, desc in state[:n]:   print(f"  {desc:<55} {score:+.4f}")

print_top_transitions(crf_model)
print_top_state_features(crf_model)

---
## 5. Model 2 — BiLSTM
### 5.1 Dataset & DataLoader PyTorch

In [ ]:
class NERDataset(Dataset):
    """Dataset NER PyTorch. Input: list of (tokens, labels). Output: (token_ids, label_ids, length)."""
    def __init__(self, data, word2idx, label2idx, pad_idx=0, pad_label_idx=32, max_len=None):
        self.pad_idx       = pad_idx
        self.pad_label_idx = pad_label_idx
        self.samples       = []
        lengths            = [len(tokens) for tokens, _ in data]
        self.max_len       = max_len or max(lengths)

        for tokens, labels in data:
            tok_ids = [word2idx.get(t, UNK_IDX) for t in tokens]
            lab_ids = [label2idx[l] for l in labels]
            self.samples.append((tok_ids, lab_ids, len(tok_ids)))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        tok_ids, lab_ids, length = self.samples[idx]
        pad_len = self.max_len - length
        tok_ids = tok_ids + [self.pad_idx]       * pad_len
        lab_ids = lab_ids + [self.pad_label_idx] * pad_len
        return (torch.tensor(tok_ids, dtype=torch.long),
                torch.tensor(lab_ids, dtype=torch.long),
                torch.tensor(length,  dtype=torch.long))


def collate_sorted(batch):
    """Sort descending by length untuk pack_padded_sequence."""
    batch   = sorted(batch, key=lambda x: x[2], reverse=True)
    tokens  = torch.stack([b[0] for b in batch])
    labels  = torch.stack([b[1] for b in batch])
    lengths = torch.stack([b[2] for b in batch])
    return tokens, labels, lengths


train_dataset = NERDataset(train_data, word2idx, label2idx, PAD_IDX, PAD_LABEL_IDX)
dev_dataset   = NERDataset(dev_data,   word2idx, label2idx, PAD_IDX, PAD_LABEL_IDX,
                            max_len=train_dataset.max_len)
test_dataset  = NERDataset(test_data,  word2idx, label2idx, PAD_IDX, PAD_LABEL_IDX,
                            max_len=train_dataset.max_len)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_sorted)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_sorted)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_sorted)

# Ambil satu batch untuk verifikasi
tokens_b, labels_b, lengths_b = next(iter(train_loader))
print(f"Train sequences : {len(train_dataset):,}")
print(f"Dev sequences   : {len(dev_dataset):,}")
print(f"Test sequences  : {len(test_dataset):,}")
print(f"Batch shape     : tokens={tokens_b.shape} | labels={labels_b.shape}")
print("✅ DataLoader siap.")

### 5.2 Arsitektur BiLSTM

In [ ]:
class BiLSTMTagger(nn.Module):
    """
    BiLSTM untuk sequence labeling (NER).
    Alur: Token IDs → Embedding → Dropout → BiLSTM → Dropout → Linear → LogSoftmax
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers, num_labels, dropout, pad_idx):
        super().__init__()
        self.embedding   = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm        = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=num_layers,
                                   bidirectional=True, batch_first=True,
                                   dropout=dropout if num_layers > 1 else 0.0)
        self.dropout     = nn.Dropout(dropout)
        self.fc          = nn.Linear(hidden_dim, num_labels)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, tokens, lengths):
        T_in    = tokens.size(1)
        emb     = self.dropout(self.embedding(tokens))
        packed  = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out_p, _= self.lstm(packed)
        out, _  = pad_packed_sequence(out_p, batch_first=True, total_length=T_in)
        out     = self.dropout(out)
        return self.log_softmax(self.fc(out))          # (B, T, num_labels)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


bilstm_model = BiLSTMTagger(
    vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, num_labels=NUM_LABELS, dropout=DROPOUT, pad_idx=PAD_IDX,
).to(DEVICE)

print(f"Parameter trainable: {count_parameters(bilstm_model):,}")

# Verifikasi forward pass
bilstm_model.eval()
with torch.no_grad():
    out_check = bilstm_model(tokens_b[:2].to(DEVICE), lengths_b[:2])
    assert tokens_b[:2].shape[1] == out_check.shape[1], "Shape tidak konsisten!"
    print(f"Input  shape : {tokens_b[:2].shape}")
    print(f"Output shape : {out_check.shape}  ← harus (2, T, {NUM_LABELS})")
    print("✅ Arsitektur BiLSTM OK.")

### 5.3 Training Utilities BiLSTM

In [ ]:
def build_weighted_criterion(train_loader, idx2label, pad_label_idx, num_labels, device,
                              o_weight=0.3, max_weight=10.0):
    """Weighted NLLLoss: label langka diberi bobot lebih tinggi, label 'O' diturunkan."""
    label_counts = Counter()
    for _, labels, lengths in train_loader:
        for lab_seq, length in zip(labels, lengths):
            for l in lab_seq[:length.item()]:
                if l.item() != pad_label_idx:
                    label_counts[idx2label[l.item()]] += 1
    total   = sum(label_counts.values())
    weights = torch.ones(num_labels)
    for idx in range(num_labels):
        label = idx2label[idx]
        if label == 'O':
            weights[idx] = o_weight
        elif label in label_counts:
            weights[idx] = min(total / (num_labels * label_counts[label]), max_weight)
    return nn.NLLLoss(weight=weights.to(device), ignore_index=pad_label_idx)


def train_one_epoch_bilstm(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for tokens, labels, lengths in loader:
        tokens, labels = tokens.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        log_probs = model(tokens, lengths)
        B, T, L   = log_probs.shape
        loss      = criterion(log_probs.view(B*T, L), labels.view(B*T))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_bilstm(model, loader, criterion, idx2label, pad_label_idx):
    model.eval()
    total_loss, all_true, all_pred = 0, [], []
    with torch.no_grad():
        for tokens, labels, lengths in loader:
            tokens, labels = tokens.to(DEVICE), labels.to(DEVICE)
            log_probs = model(tokens, lengths)
            B, T, L   = log_probs.shape
            total_loss += criterion(log_probs.view(B*T, L), labels.view(B*T)).item()
            preds = log_probs.argmax(dim=-1)
            for pred_seq, true_seq, length in zip(preds, labels, lengths):
                l = length.item()
                all_pred.append([idx2label[p.item()] for p in pred_seq[:l]])
                all_true.append([idx2label[t.item()] for t in true_seq[:l]
                                  if t.item() != pad_label_idx])
    f1 = seq_f1(all_true, all_pred, zero_division=0)
    return total_loss / len(loader), f1, all_true, all_pred


criterion_bilstm = build_weighted_criterion(
    train_loader, idx2label, PAD_LABEL_IDX, NUM_LABELS, DEVICE
)
print("✅ Training utilities BiLSTM siap.")

### 5.4 Pelatihan BiLSTM

In [ ]:
# Reset model & optimizer menggunakan BILSTM_LR dari konfigurasi global
bilstm_model = BiLSTMTagger(
    vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, num_labels=NUM_LABELS, dropout=DROPOUT, pad_idx=PAD_IDX,
).to(DEVICE)

optimizer_bilstm = optim.AdamW(bilstm_model.parameters(),
                                lr=BILSTM_LR, weight_decay=WEIGHT_DECAY)
scheduler_bilstm = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_bilstm, mode='max', factor=0.5, patience=5
)

best_f1, patience_cnt, best_state = 0.0, 0, None
bilstm_history = {'train_loss': [], 'dev_loss': [], 'dev_f1': []}

print(f"{'='*60}\n Melatih BiLSTM\n{'='*60}")
print(f" Vocab size    : {VOCAB_SIZE:,}")
print(f" Embedding dim : {EMBEDDING_DIM} | Hidden dim : {HIDDEN_DIM}")
print(f" Num layers    : {NUM_LAYERS}   | Dropout    : {DROPOUT}")
print(f" Learning rate : {BILSTM_LR}   | Batch size : {BATCH_SIZE}")
print(f" Max epochs    : {MAX_EPOCHS}   | Patience   : {PATIENCE}")
print(f"{'='*60}")

t_start = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss           = train_one_epoch_bilstm(bilstm_model, train_loader, optimizer_bilstm, criterion_bilstm)
    dev_loss, dev_f1, _, _ = evaluate_bilstm(bilstm_model, dev_loader, criterion_bilstm, idx2label, PAD_LABEL_IDX)
    scheduler_bilstm.step(dev_f1)

    bilstm_history['train_loss'].append(train_loss)
    bilstm_history['dev_loss'].append(dev_loss)
    bilstm_history['dev_f1'].append(dev_f1)

    print(f"  Epoch {epoch:3d}/{MAX_EPOCHS} | Train Loss: {train_loss:.4f} | "
          f"Dev Loss: {dev_loss:.4f} | Dev F1: {dev_f1:.4f}", end='')

    if dev_f1 > best_f1:
        best_f1      = dev_f1
        best_state   = {k: v.cpu().clone() for k, v in bilstm_model.state_dict().items()}
        patience_cnt = 0
        print(" ✓ best")
    else:
        patience_cnt += 1
        print()
    if patience_cnt >= PATIENCE:
        print(f"  Early stopping pada epoch {epoch}.")
        break

elapsed_bilstm = time.time() - t_start
bilstm_model.load_state_dict(best_state)
print(f"\nTotal waktu latih: {elapsed_bilstm:.1f} detik | Best Dev F1: {best_f1:.4f}")

### 5.5 Evaluasi BiLSTM pada Test Set

In [ ]:
_, _, y_true_bilstm, y_pred_bilstm = evaluate_bilstm(
    bilstm_model, test_loader, criterion_bilstm, idx2label, PAD_LABEL_IDX
)

print("="*60)
print(" HASIL EVALUASI — BiLSTM (Test Set)")
print("="*60)
print(seq_classification_report(y_true_bilstm, y_pred_bilstm, digits=4))

bilstm_results = {
    'model':      'BiLSTM',
    'precision':  seq_precision(y_true_bilstm, y_pred_bilstm),
    'recall':     seq_recall(y_true_bilstm, y_pred_bilstm),
    'f1':         seq_f1(y_true_bilstm, y_pred_bilstm),
    'train_time': elapsed_bilstm,
}
print(f"Macro P/R/F1 : {bilstm_results['precision']:.4f} / {bilstm_results['recall']:.4f} / {bilstm_results['f1']:.4f}")
print(f"Waktu latih  : {elapsed_bilstm:.1f} detik")

---
## 6. Model 3 — BiLSTM-CRF
### 6.1 Arsitektur BiLSTM-CRF

> BiLSTM-CRF **berbagi layer Embedding + BiLSTM** yang sama dengan BiLSTM murni.
> Perbedaannya: output Linear tidak langsung ke Softmax, melainkan ke **CRF layer**
> yang memodelkan dependensi antar label (transition matrix) menggunakan Viterbi decoding.

In [ ]:
class BiLSTMCRFTagger(nn.Module):
    """
    BiLSTM-CRF untuk sequence labeling.
    Alur: Token IDs → Embedding → Dropout → BiLSTM → Dropout → Linear (emissions) → CRF
    Menggunakan torchcrf (pytorch-crf) untuk CRF layer yang efisien.
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers, num_labels, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm      = nn.LSTM(embedding_dim, hidden_dim // 2, num_layers=num_layers,
                                  bidirectional=True, batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, num_labels)
        self.crf       = CRF(num_labels, batch_first=True)

    def _get_emissions(self, tokens, lengths):
        T_in    = tokens.size(1)
        emb     = self.dropout(self.embedding(tokens))
        packed  = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=True)
        out_p, _= self.lstm(packed)
        out, _  = pad_packed_sequence(out_p, batch_first=True, total_length=T_in)
        return self.fc(self.dropout(out))                  # (B, T, num_labels)

    def _make_mask(self, lengths, max_len):
        return (torch.arange(max_len, device=lengths.device).unsqueeze(0)
                < lengths.to(lengths.device).unsqueeze(1))

    def forward(self, tokens, lengths, labels=None):
        emissions   = self._get_emissions(tokens, lengths)
        lengths_dev = lengths.to(tokens.device)
        mask        = self._make_mask(lengths_dev, tokens.size(1))

        if labels is not None:
            # torchcrf tidak punya ignore_index — ganti PAD dengan 'O' (idx=0)
            labels_crf = labels.clone()
            labels_crf[labels_crf == PAD_LABEL_IDX] = 0
            return -self.crf(emissions, labels_crf, mask=mask, reduction='mean')
        else:
            return self.crf.decode(emissions, mask=mask)


bilstm_crf_model = BiLSTMCRFTagger(
    vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, num_labels=NUM_LABELS, dropout=DROPOUT, pad_idx=PAD_IDX,
).to(DEVICE)

print(f"Parameter trainable: {count_parameters(bilstm_crf_model):,}")

# Verifikasi forward pass
bilstm_crf_model.eval()
with torch.no_grad():
    tok_s  = tokens_b[:2].to(DEVICE)
    lab_s  = labels_b[:2].to(DEVICE)
    len_s  = lengths_b[:2]
    loss_v = bilstm_crf_model(tok_s, len_s, labels=lab_s)
    pred_v = bilstm_crf_model(tok_s, len_s)
    print(f"Forward (train) Loss : {loss_v.item():.4f}")
    print(f"Forward (infer) Pred : {pred_v[0][:5]}")
    print("✅ Arsitektur BiLSTM-CRF OK.")

### 6.2 Training Utilities BiLSTM-CRF

In [ ]:
def train_one_epoch_crf(model, loader, optimizer):
    """Training epoch BiLSTM-CRF — loss langsung dari CRF (NLL)."""
    model.train()
    total_loss = 0
    for tokens, labels, lengths in loader:
        tokens, labels = tokens.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = model(tokens, lengths, labels=labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_bilstm_crf(model, loader, idx2label, pad_label_idx):
    """Evaluasi BiLSTM-CRF menggunakan Viterbi decoding dari CRF."""
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for tokens, labels, lengths in loader:
            tokens, labels = tokens.to(DEVICE), labels.to(DEVICE)
            preds = model(tokens, lengths)               # list of list
            for pred_seq, true_seq, length in zip(preds, labels, lengths):
                l = length.item()
                all_pred.append([idx2label[p] for p in pred_seq[:l]])
                all_true.append([idx2label[t.item()] for t in true_seq[:l]
                                  if t.item() != pad_label_idx])
    f1 = seq_f1(all_true, all_pred, zero_division=0)
    return f1, all_true, all_pred


print("✅ Training utilities BiLSTM-CRF siap.")

### 6.3 Pelatihan BiLSTM-CRF

In [ ]:
# Reset model & optimizer menggunakan BILSTMCRF_LR dari konfigurasi global
bilstm_crf_model = BiLSTMCRFTagger(
    vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS, num_labels=NUM_LABELS, dropout=DROPOUT, pad_idx=PAD_IDX,
).to(DEVICE)

optimizer_crf = optim.AdamW([
    {
        'params': bilstm_crf_model.embedding.parameters(),
        'lr': 1e-4,           # sama dengan LSTM
        'name': 'embedding'
    },
    {
        'params': list(bilstm_crf_model.lstm.parameters()) +
                  list(bilstm_crf_model.fc.parameters()),
        'lr': 3e-4,           # LSTM + Linear emission
        'name': 'lstm+fc'
    },
    {
        'params': bilstm_crf_model.crf.parameters(),
        'lr': 2e-5,           # CRF: setengah dari LSTM
        'name': 'crf'
    },
], weight_decay=WEIGHT_DECAY)
scheduler_crf = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_crf, mode='max', factor=0.5, patience=5
)

best_f1_crf, patience_cnt_crf, best_state_crf = 0.0, 0, None
bilstm_crf_history = {'train_loss': [], 'dev_f1': []}

print(f"{'='*60}\n Melatih BiLSTM-CRF\n{'='*60}")
print(f" Vocab size    : {VOCAB_SIZE:,}")
print(f" Embedding dim : {EMBEDDING_DIM} | Hidden dim : {HIDDEN_DIM}")
print(f" Num layers    : {NUM_LAYERS}   | Dropout    : {DROPOUT}")
print(f" Learning rate : {BILSTMCRF_LR} | Batch size : {BATCH_SIZE}")
print(f" Max epochs    : {MAX_EPOCHS}   | Patience   : {PATIENCE}")
print(f"{'='*60}")

t_start_crf = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss          = train_one_epoch_crf(bilstm_crf_model, train_loader, optimizer_crf)
    dev_f1, _, _        = evaluate_bilstm_crf(bilstm_crf_model, dev_loader, idx2label, PAD_LABEL_IDX)
    scheduler_crf.step(dev_f1)

    bilstm_crf_history['train_loss'].append(train_loss)
    bilstm_crf_history['dev_f1'].append(dev_f1)

    print(f"  Epoch {epoch:3d}/{MAX_EPOCHS} | Train Loss: {train_loss:.4f} | Dev F1: {dev_f1:.4f}", end='')

    if dev_f1 > best_f1_crf:
        best_f1_crf      = dev_f1
        best_state_crf   = {k: v.cpu().clone() for k, v in bilstm_crf_model.state_dict().items()}
        patience_cnt_crf = 0
        print(" ✓ best")
    else:
        patience_cnt_crf += 1
        print()
    if patience_cnt_crf >= PATIENCE:
        print(f"  Early stopping pada epoch {epoch}.")
        break

elapsed_crf = time.time() - t_start_crf
bilstm_crf_model.load_state_dict(best_state_crf)
print(f"\nTotal waktu latih: {elapsed_crf:.1f} detik | Best Dev F1: {best_f1_crf:.4f}")

### 6.4 Evaluasi BiLSTM-CRF pada Test Set

In [ ]:
_, y_true_bilstm_crf, y_pred_bilstm_crf = evaluate_bilstm_crf(
    bilstm_crf_model, test_loader, idx2label, PAD_LABEL_IDX
)

print("="*60)
print(" HASIL EVALUASI — BiLSTM-CRF (Test Set)")
print("="*60)
print(seq_classification_report(y_true_bilstm_crf, y_pred_bilstm_crf, digits=4))

bilstm_crf_results = {
    'model':      'BiLSTM-CRF',
    'precision':  seq_precision(y_true_bilstm_crf, y_pred_bilstm_crf),
    'recall':     seq_recall(y_true_bilstm_crf, y_pred_bilstm_crf),
    'f1':         seq_f1(y_true_bilstm_crf, y_pred_bilstm_crf),
    'train_time': elapsed_crf,
}
print(f"Macro P/R/F1 : {bilstm_crf_results['precision']:.4f} / {bilstm_crf_results['recall']:.4f} / {bilstm_crf_results['f1']:.4f}")
print(f"Waktu latih  : {elapsed_crf:.1f} detik")

---
## 7. Perbandingan & Visualisasi Hasil

In [ ]:
all_results = [crf_results, bilstm_results, bilstm_crf_results]
df_results  = pd.DataFrame(all_results).set_index('model')
df_results.columns = ['Precision', 'Recall', 'F1', 'Train Time (s)']

print("\n" + "="*60)
print(" PERBANDINGAN SEMUA MODEL (Test Set)")
print("="*60)
print(df_results.to_string(float_format=lambda x: f'{x:.4f}'))

best_model = df_results['F1'].idxmax()
print(f"\n🏆 Model terbaik berdasarkan F1: {best_model} ({df_results.loc[best_model,'F1']:.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Baseline Model Performance Comparison', fontsize=13, fontweight='bold')

models  = df_results.index.tolist()
metrics = ['Precision', 'Recall', 'F1']
colors  = ['#4C72B0', '#DD8452', '#55A868']
x, width = np.arange(len(models)), 0.25

ax = axes[0]
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = df_results[metric].values
    bars = ax.bar(x + i*width, vals, width, label=metric, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xlabel('Model'); ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 (micro seqeval)')
ax.set_xticks(x + width); ax.set_xticklabels(models)
ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
times = df_results['Train Time (s)'].values
bars2 = ax2.bar(models, times, color=colors, alpha=0.85)
for bar, val in zip(bars2, times):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.0f}s', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('second'); ax2.set_title('Training Time (seconds)'); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik disimpan ke baseline_comparison.png")

In [ ]:
# Learning Curves BiLSTM & BiLSTM-CRF
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Learning Curves', fontsize=13, fontweight='bold')

for ax, hist, name in zip(axes,
    [bilstm_history, bilstm_crf_history], ['BiLSTM', 'BiLSTM-CRF']):
    ep = range(1, len(hist['train_loss'])+1)
    ax.plot(ep, hist['train_loss'], label='Train Loss', color='#4C72B0')
    if 'dev_loss' in hist:
        ax.plot(ep, hist['dev_loss'], label='Dev Loss', color='#DD8452', linestyle='--')
    ax2_ = ax.twinx()
    ax2_.plot(ep, hist['dev_f1'], label='Dev F1', color='#55A868', linewidth=2)
    ax2_.set_ylabel('Dev F1', color='#55A868'); ax2_.set_ylim(0, 1)
    ax.set_title(name); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2_.get_legend_handles_labels()
    ax.legend(lines1+lines2, labels1+labels2, loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Analisis Per-Label (Test Set)

In [ ]:
def per_label_df(y_true, y_pred, model_name):
    report = seq_classification_report(y_true, y_pred, output_dict=True, digits=4)
    rows = []
    for label, scores in report.items():
        if label in ('micro avg', 'macro avg', 'weighted avg'):
            continue
        rows.append({'Label': label, 'Precision': scores['precision'],
                     'Recall': scores['recall'], 'F1': scores['f1-score'],
                     'Support': scores['support'], 'Model': model_name})
    return pd.DataFrame(rows)


df_crf       = per_label_df(y_test, y_test_pred_crf,      'CRF')
df_bilstm    = per_label_df(y_true_bilstm, y_pred_bilstm, 'BiLSTM')
df_bilstmcrf = per_label_df(y_true_bilstm_crf, y_pred_bilstm_crf, 'BiLSTM-CRF')

pivot_f1 = (pd.concat([df_crf, df_bilstm, df_bilstmcrf])
              .pivot(index='Label', columns='Model', values='F1')
              .fillna(0))
pivot_f1['Support'] = df_crf.set_index('Label')['Support']

print("\nF1 Per Label (Test Set):")
print(pivot_f1.sort_values('BiLSTM-CRF', ascending=False)
              .to_string(float_format=lambda x: f'{x:.4f}'))

In [ ]:
import matplotlib.colors as mcolors

pivot_plot = pivot_f1[['CRF', 'BiLSTM', 'BiLSTM-CRF']].sort_values('BiLSTM-CRF', ascending=False)
fig, ax = plt.subplots(figsize=(8, max(5, len(pivot_plot)*0.45)))
im = ax.imshow(pivot_plot.values, cmap='YlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(3)); ax.set_xticklabels(['CRF', 'BiLSTM', 'BiLSTM-CRF'], fontsize=11)
ax.set_yticks(range(len(pivot_plot))); ax.set_yticklabels(pivot_plot.index, fontsize=9)

for i in range(len(pivot_plot)):
    for j in range(3):
        val   = pivot_plot.values[i, j]
        color = 'black' if val < 0.6 else 'white'
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='F1 Score')
ax.set_title('F1 Per Label — Perbandingan Tiga Baseline', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('f1_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Simpan Model & Hasil

In [ ]:
import pickle
os.makedirs('saved_models_6', exist_ok=True)

# CRF
with open('saved_models_6/crf_model.pkl', 'wb') as f:
    pickle.dump(crf_model, f)

# BiLSTM
torch.save({
    'model_state': bilstm_model.state_dict(),
    'config': {'vocab_size': VOCAB_SIZE, 'embedding_dim': EMBEDDING_DIM,
               'hidden_dim': HIDDEN_DIM,  'num_layers': NUM_LAYERS,
               'num_labels': NUM_LABELS,  'dropout': DROPOUT, 'pad_idx': PAD_IDX},
    'word2idx': word2idx, 'label2idx': label2idx,
}, 'saved_models_6/bilstm_model.pt')

# BiLSTM-CRF
torch.save({
    'model_state': bilstm_crf_model.state_dict(),
    'config': {'vocab_size': VOCAB_SIZE, 'embedding_dim': EMBEDDING_DIM,
               'hidden_dim': HIDDEN_DIM,  'num_layers': NUM_LAYERS,
               'num_labels': NUM_LABELS,  'dropout': DROPOUT, 'pad_idx': PAD_IDX},
    'word2idx': word2idx, 'label2idx': label2idx,
}, 'saved_models_6/bilstm_crf_model.pt')

# Hasil CSV & JSON
df_results.to_csv('saved_models_6/baseline_results.csv')
with open('saved_models_6/baseline_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("✅ Semua model dan hasil tersimpan di ./saved_models_5/")

---
## 10. Inference Demo

In [ ]:
def predict_sentence(text_or_tokens, model_choice='bilstm_crf'):
    """
    Prediksi entitas dari teks / list token.
    model_choice: 'crf' | 'bilstm' | 'bilstm_crf'
    """
    tokens = text_or_tokens.split() if isinstance(text_or_tokens, str) else text_or_tokens

    if model_choice == 'crf':
        preds = crf_model.predict_single(sent2features(tokens))

    else:
        t_ids    = [word2idx.get(t, UNK_IDX) for t in tokens]
        t_tensor = torch.tensor([t_ids], dtype=torch.long).to(DEVICE)
        lengths  = torch.tensor([len(t_ids)], dtype=torch.long)

        if model_choice == 'bilstm':
            bilstm_model.eval()
            with torch.no_grad():
                preds = [idx2label[p] for p in bilstm_model(t_tensor, lengths).argmax(-1)[0].tolist()]
        else:
            bilstm_crf_model.eval()
            with torch.no_grad():
                preds = [idx2label[p] for p in bilstm_crf_model(t_tensor, lengths)[0]]

    print(f"\n{'─'*55}\n Model: {model_choice.upper()}\n{'─'*55}")
    print(f"{'TOKEN':<25} {'LABEL'}")
    print(f"{'─'*55}")
    for tok, lbl in zip(tokens, preds):
        print(f"  {tok:<23} {lbl}{'  ←' if lbl != 'O' else ''}")
    return list(zip(tokens, preds))


contoh = [
    "Dr. Anwar Usman S.H. M.H. sebagai Ketua merangkap Anggota",
    "Putusan Nomor 91/PUU-XX/2022 diucapkan pada tanggal 5 Oktober 2022",
    "Menimbang bahwa Mahkamah Konstitusi berwenang mengadili perkara",
]
for kalimat in contoh:
    predict_sentence(kalimat, 'bilstm_crf')

---
## 11. Ringkasan Eksperimen

In [ ]:
print("\n" + "#"*65)
print("  RINGKASAN EKSPERIMEN — LER BASELINE PUTUSAN PUU MKRI")
print("#"*65)
print(f"""
Corpus   : Putusan PUU Mahkamah Konstitusi RI
Anotasi  : doccano → BIO scheme
Labels   : {NUM_LABELS} label
Split    : Train / Dev / Test

Hyperparameter (BiLSTM & BiLSTM-CRF):
  Embedding dim  : {EMBEDDING_DIM}
  Hidden dim     : {HIDDEN_DIM}
  Layers         : {NUM_LAYERS}
  Dropout        : {DROPOUT}
  Batch size     : {BATCH_SIZE}
  Max epochs     : {MAX_EPOCHS}
  Early stop     : {PATIENCE} epochs
  BiLSTM LR      : {BILSTM_LR}
  BiLSTM-CRF LR  : {BILSTMCRF_LR}
  Weight decay   : {WEIGHT_DECAY}
""")

print("-"*65)
print(f"{'Model':<14} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Time(s)':>10}")
print("-"*65)
for r in all_results:
    print(f"{r['model']:<14} {r['precision']:>10.4f} {r['recall']:>10.4f} "
          f"{r['f1']:>10.4f} {r['train_time']:>10.1f}")
print("-"*65)
print(f"\n🏆 Model terbaik: {best_model}")